In [1]:
import base64
import json
import sqlite3
from datetime import datetime

In [2]:
# Extract
RAW_JSON_PAYLOAD = """
{
    "record_id": "TX-1092",
    "timestamp": "2026-07-16T03:44:00Z",
    "encoded_data": "eyJuYW1lIjogIkpvaG4gRG9lIiwgInN0YXR1cyI6ICJhY3RpdmUifQ==",
    "encoded_file": "SGVsbG8gV29ybGQh" 
}
"""

In [ ]:
def etl_pipeline(json_string):
    try:
        # load raw JSON - parse into python object
        raw_data = json.loads(json_string)
        print(f"Raw data: {raw_data}")
        print("[x] Data Extracted")

        # transform: decode nested JSON string
        decoded_bytes = base64.b64decode(raw_data['encoded_data']) # converts the base64 back to raw bytes
        decoded_json = json.loads(decoded_bytes.decode('utf-8')) # turns raw bytes into usable text string, then turns it into dictionary

        print(f"Decoded JSON: {decoded_json}")

        # transform: decode the file payload
        file_content = base64.b64decode(raw_data['encoded_file']).decode('utf-8')

        print(f"Decoded file: {file_content}")

        transformed_payload = {
            "record_id": raw_data['record_id'],
            "timestamp": datetime.fromisoformat(raw_data['timestamp'].replace('Z', '+00:00')),
            "user_name": decoded_json.get('name'),
            "user_status": decoded_json.get('status'),
            "file_preview": file_content
        }
        print("[x] Data Transformed")

        conn = sqlite3.connect('etl_pipeline.db')
        cursor = conn.cursor()

        cursor.execute('''
            CREATE TABLE IF NOT EXISTS parsed_records (
                record_id TEXT PRIMARY KEY,
                timestamp TEXT,
                user_name TEXT,
                user_status TEXT,
                file_preview TEXT
            )
        ''')

        cursor.execute('''
            INSERT OR REPLACE INTO parsed_records 
            (record_id, timestamp, user_name, user_status, file_preview)
            VALUES (?, ?, ?, ?, ?)
        ''', (
            transformed_payload['record_id'],
            transformed_payload['timestamp'].isoformat(),
            transformed_payload['user_name'],
            transformed_payload['user_status'],
            transformed_payload['file_preview']
        ))

        data = cursor.execute('''SELECT * FROM parsed_records''')
        for item in data:
            print(item)

        conn.commit()
        conn.close()
        print("[x] Data successfully loaded")

    except Exception as e:
        print(f"Error processing pipeline: {e}")

In [19]:
etl_pipeline(RAW_JSON_PAYLOAD)

Raw data: {'record_id': 'TX-1092', 'timestamp': '2026-07-16T03:44:00Z', 'encoded_data': 'eyJuYW1lIjogIkpvaG4gRG9lIiwgInN0YXR1cyI6ICJhY3RpdmUifQ==', 'encoded_file': 'SGVsbG8gV29ybGQh'}
[x] Data Extracted
Decoded JSON: {'name': 'John Doe', 'status': 'active'}
Decoded file: Hello World!
[x] Data Transformed
('TX-1092', '2026-07-16T03:44:00+00:00', 'John Doe', 'active', 'Hello World!')
[x] Data successfully loaded
